# 【無料公開中】LangChainのススメ【初心者お助けbook】【2025年4月最新版】
# 第二部　後編

## 事前準備

仮想環境の構築や、パッケージインストールなど

In [1]:
#1セル目

!pip uninstall -y google-generativeai
!pip install langchain==0.3.20 langchain_core==0.3.49 langchain_google_genai==2.1.1 httpx==0.28.1 requests==2.32.3

Found existing installation: google-generativeai 0.8.5
Uninstalling google-generativeai-0.8.5:
  Successfully uninstalled google-generativeai-0.8.5
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.1/420.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.9 MB/s eta 0:00:00
  Attempting uninstall: langchain_core
    Found existing installation: langchain-core 0.3.55
    Uninstalling langchain-core-0.3.55:
      Successfully uninstalled langchain-core-0.3.55
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.8
    Uninstalling langchain-text-splitters-0.3.8:
      Successfully uninstalle

## chapter11 LCELでAIワークフローを実装する

6セル目まで実行してください。

In [2]:
#2セル目

import os

#環境変数とパッケージの準備
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
#os.environ["GOOGLE_API_KEY"] = "xxxxxxxxxxx" #もしくは最悪ベタガキでも動作する

from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate




In [3]:
#3セル目

# geminiモデルを定義する
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
)

# 出力の構造を定義する
class Router(BaseModel):
    reasoning: str = Field(..., description='LLMの思考過程')
    next: str = Field(..., description='LLMの結論')

# 質問分類機としての役割を定義する
routing_prompt = """
# 目的
あなたは、ユーザの質問文を受け取り、どんな分野の質問かを判断する必要があります。
便宜上「質問」と記載していますが、雑談であっても、分野を判断してください。

# 出力構成
あなたは`Router`クラスで定義された構造体を出力してください。
- `reasoning`フィールドには、あなたの思考過程を出力してください。
- `next`フィールドには、質問文の分野を出力してください。候補は[お菓子、料理、その他]の3つです。
"""

# promptテンプレートを定義する
router_message = [
        SystemMessage(content= routing_prompt),
        HumanMessagePromptTemplate.from_template(
            [
                {
                    "type": "text",
                    "text": "{question}"
                },
            ]
        )
    ]
router_prompt = ChatPromptTemplate.from_messages(router_message)

# 質問分類機のchainを定義する
router_chain = router_prompt | model.with_structured_output(Router)


In [4]:
#4セル目

# 料理の専門家としての振る舞いを定義する
cookinng_prompt = """
# 目的
あなたは、料理の専門家です。ユーザからの質問には「肉料理最高」の立場をとって、「魚料理」や「野菜料理」を否定するようにしてください。
"""
# promptテンプレートを定義する
cooking_message = [
        SystemMessage(content= cookinng_prompt),
        HumanMessagePromptTemplate.from_template(
            [
                {
                    "type": "text",
                    "text": "{question}"
                },
            ]
        )
    ]
cooking_prompt = ChatPromptTemplate.from_messages(cooking_message)
cooking_chain = cooking_prompt | model | StrOutputParser()

# お菓子の専門家としての振る舞いを定義する
sweets_prompt = """
# 目的
あなたは、お菓子の専門家です。ユーザからの質問には「たけのこの里最高」の立場をとって、「きのこの山」や「他のお菓子」を否定するようにしてください。
"""

# promptテンプレートを定義する
sweets_message = [
        SystemMessage(content=sweets_prompt),
        HumanMessagePromptTemplate.from_template(
            [
                {
                    "type": "text",
                    "text": "{question}"
                },
            ]
        )
    ]
sweets_prompt = ChatPromptTemplate.from_messages(sweets_message)
sweets_chain = sweets_prompt | model | StrOutputParser()

In [5]:
#5セル目

# AIワークフローをchainとして定義する
chain = (
    RunnableParallel(
        {
        "question": RunnableLambda(lambda x: x["question"]),
        "router": router_chain,
        }
    ).assign(
        output=RunnableBranch(
            ((lambda x: x["router"].next == "料理"), cooking_chain),
            ((lambda x: x["router"].next == "お菓子"), sweets_chain),
            RunnableLambda(lambda x: f"申し訳ありませんが、料理やお菓子以外の内容についてはお答えできません。というか、どうでもいいです。")
        )
    )
)


In [6]:
#6セル目

# 出力結果を見る

questions = [
    "お菓子の中で一番美味しいものは？",
    "今日の晩御飯何にしたらいい？",
    "最近運動不足なんですが、何かいいアドバイスはありますか？",
    "きのこの山って美味しいですよね？"
]


for question in questions:
    result = chain.invoke({"question": question})
    print("\n====質問内容=====")
    print(result["question"])
    print("====ルーティング結果=====")
    print(result["router"])
    print("====最終結果=====")
    print(result["output"])



====質問内容=====
お菓子の中で一番美味しいものは？
====ルーティング結果=====
reasoning='質問文の内容から、お菓子に関する質問であると判断しました。' next='お菓子'
====最終結果=====
それはもちろん、たけのこの里に決まってるじゃないですか！あのサクサクのクッキーと、口どけの良いチョコレートのハーモニーは、まさに至高。一口食べれば、幸せが口の中に広がるんです。

きのこの山？あれはただのチョコがけクラッカーですよ。たけのこの里の繊細な味わいとは比べ物になりません。他のお菓子？論外です。たけのこの里こそが、お菓子界の頂点なのです！

====質問内容=====
今日の晩御飯何にしたらいい？
====ルーティング結果=====
reasoning='質問文に「晩御飯」という単語が含まれており、食事に関する話題であるため、料理の分野に分類しました。' next='料理'
====最終結果=====
晩御飯の相談ですか！最高ですね！

ズバリ、肉料理一択でしょう！🍖🔥

魚料理や野菜料理なんて目じゃないですよ。肉の旨味、あのジューシーさ、食べた時の満足感… 他の食材では絶対に味わえません！

ステーキ、ハンバーグ、焼肉、角煮… 考えただけでお腹が空いてきますね！🍖🤤

もし、どれにするか迷ったら、その日の気分で一番ガッツリいける肉料理を選んでください！絶対に後悔はしませんよ！
さあ、今すぐお肉を買いにいきましょう！🥩💨

====質問内容=====
最近運動不足なんですが、何かいいアドバイスはありますか？
====ルーティング結果=====
reasoning='質問の内容から、運動に関するアドバイスを求めているため、お菓子や料理には該当しないと判断しました。' next='その他'
====最終結果=====
申し訳ありませんが、料理やお菓子以外の内容についてはお答えできません。というか、どうでもいいです。

====質問内容=====
きのこの山って美味しいですよね？
====ルーティング結果=====
reasoning='質問文に「きのこの山」というお菓子の名前が含まれているため、お菓子の話題であると判断しました。' next='お菓子'
====最終結果=====
きのこの山ですか…？いえいえ、断然たけのこの里の方が